# Importing Basic Packages

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (auc, accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             roc_auc_score, precision_recall_curve, average_precision_score)



In [ ]:
import pandas as pd

file = r'/kaggle/input/ana-verse-2-0-h/train.parquet'

df = pd.read_parquet(file)
df

# Checking for Missing Values

In [ ]:
df.info()

#### There are no missing values in the dataset

In [ ]:

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")


df["target"] = pd.to_numeric(df["target"], errors="coerce").fillna(0).astype(int)


df["month"] = df["Date"].dt.month


avg_anomaly_by_month = df.groupby("month")["target"].mean()


plt.figure(figsize=(8,4))
avg_anomaly_by_month.plot(kind="bar")
plt.title("Average Anomaly Rate by Calendar Month")
plt.ylabel("Anomaly Rate")
plt.xlabel("Month")
plt.tight_layout()
plt.show()


The anomaly rate follows a clear seasonal U-shape, peaking in January and December and reaching a minimum in mid-year, indicating strong seasonality in plant instability

In [ ]:
df["dow"] = df["Date"].dt.dayofweek  # 0=Mon

avg_anomaly_by_dow = (
    df.groupby("dow")["target"]
      .mean()
)

avg_anomaly_by_dow.plot(kind="bar")
plt.title("Average Anomaly Rate by Day of Week")
plt.ylabel("Anomaly Rate")
plt.xlabel("Day of Week")
plt.show()


Anomalies peak mid-week, suggesting operational stress accumulation rather than weekend staffing issues.

### ***Insight*** :- 
Anomalies show strong seasonal and weekly structure. They are most frequent in winter months and mid-week, indicating that failures are driven by operational stress and environmental conditions rather than random sensor noise.”Anomalies show strong seasonal and weekly structure. They are most frequent in winter months and mid-week, indicating that failures are driven by operational stress and environmental conditions rather than random sensor noise.

## Investigating Sensory Variables through Histograms, Summary Statistics and box plots

In [ ]:
Xn = df[['X1','X2','X3','X4','X5']]
Xn.describe()

In [ ]:

fig, axes = plt.subplots(nrows=5, ncols=2, figsize=(18, 25))


fig.suptitle("Histograms and Boxplots of Variables", fontsize=18, y=0.99)

for i, column in enumerate(Xn.columns):
    
    # Histogram
    sns.histplot(Xn[column],ax=axes[i, 0],bins=100,kde=True, color='black')
    axes[i, 0].set_title(f'Histogram of {column}', fontsize=12)
    
    # Boxplot
    sns.boxplot(x=Xn[column],ax=axes[i, 1],color='yellow',fliersize=3)       # reduce clutter for large data
    axes[i, 1].set_title(f'Boxplot of {column}', fontsize=12)

plt.tight_layout()
plt.subplots_adjust(top=0.96)
plt.show()


# Analyis of Sensory Variables (X1, X2, X3, X4, X5)

**X1** :- 
*X1 exhibits a stable and near-symmetric distribution with low variance, as confirmed by its summary statistics, histogram, and compact boxplot. The absence of significant outliers indicates a well-controlled physical process, making X1 a reliable baseline feature that requires no transformation beyond standard scaling.*

**X2** :- 
X2 shows moderate variability with mild skewness but remains well-bounded, with histograms and boxplots indicating limited and non-extreme deviations. This suggests responsiveness to any changes without abnormal spikes, allowing X2 to be used directly as a scaled continuous feature

**X3** :- 
*X3 is strongly right-skewed with a long upper tail, where a small number of extreme values dominate the variance, as clearly visible in its histogram and boxplot. A log transformation needs to be applied to reduce skewness and stabilize variance, enabling the model to learn proportional changes, while an extreme indicator needs to be created to explicitly capture rare but critical spikes that are highly indicative of anomalous behavior.*

**X4** :- 
*X4 mirrors X3 in exhibiting heavy right skewness and influential high-end outliers, with boxplots showing extreme deviations far beyond the upper quartile. Log transformation improves numerical stability and feature scaling, while an extreme flag preserves the anomaly signal carried by rare excursions that would otherwise be diluted during normalization. Alike X3, even  log transformation needs to be applied to X4 and an extreme indicator to be created to explicitly capture rare but critical spikes that are highly indicative of anomalous behavior.*

**X5** :- 
*X5 has a relatively compact distribution with moderate spread and limited outliers, as seen in its summary statistics and boxplot. Since it lacks heavy-tailed behavior, X5 does not require log transformation or extreme flagging and is suitably represented as a scaled continuous feature.*

# Correlation Analysis

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


Xn = df[['X1', 'X2', 'X3', 'X4', 'X5']]

corr_matrix = Xn.corr()

plt.figure(figsize=(5,4))
sns.heatmap(corr_matrix,annot=True,fmt=".2f",cmap="coolwarm",center=0,square=True)

plt.title("Sensor Correlation Heatmap")
plt.tight_layout()
plt.show()


The correlation heatmap shows that the sensor readings are largely uncorrelated with one another, indicating that each sensor captures mostly independent aspects of the plant’s operation rather than redundant information. X3 and X4 are almost completely independent of all other sensors, while X1 and X2 show only a very weak negative relationship, and X2–X5 exhibit the strongest (yet still modest) negative correlation at around −0.26. Overall, the absence of strong correlations suggests there is no multicollinearity concern and supports the use of tree-based ensemble models, as anomalies are likely driven by subtle, non-linear and multivariate pattern changes or by correlation breakdowns during abnormal conditions rather than by simple linear relationships between sensors.

# Correlation Analysis of each Class

In [ ]:
corr_normal  = df[df["target"] == 0][['X1','X2','X3','X4','X5']].corr()
corr_anomaly = df[df["target"] == 1][['X1','X2','X3','X4','X5']].corr()

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
sns.heatmap(corr_normal, annot=True, cmap="coolwarm", center=0, square=True)
plt.title("Correlation – Normal")

plt.subplot(1,2,2)
sns.heatmap(corr_anomaly, annot=True, cmap="coolwarm", center=0, square=True)
plt.title("Correlation – Anomaly")

plt.tight_layout()
plt.show()



The correlation analysis shows that under normal operation the sensors maintain stable, though weak, relationships (for example, X2 and X5 have a consistent negative correlation), whereas during anomalies these relationships partially break down or shift, such as the weakening of the X2–X5 inverse relationship and the emergence of a mild coupling between X2 and X4. Importantly, no strong new correlations appear in the anomalous state, indicating that anomalies are not caused by a single sensor behaving abnormally but by subtle, multivariate changes in how sensors interact. This suggests the system failures are driven by process-level instability rather than isolated sensor faults, which explains why tree-based ensemble models that can capture interaction changes perform well, **while simple linear or distance-based models are less effective.**


# Outliers
Outliers are not removed because, in an anomaly detection problem, those extreme values often are the anomalies we want to detect, and they are not noise. Removing them would eliminate the most informative signals of abnormal behavior and make the model blind to rare but critical failure conditions.

# Feature Engineering

### Creation of X3_extreme and X4_extreme variables

X3_extreme and X4_extreme are binary flags that indicate whether sensor-3 or sensor-4 is operating at an unusually high level compared to normal historical behavior. A value of 1 means the reading is exceptionally high and potentially abnormal; 0 means it is within the normal range.

In the data, I found the value below which 99% of readings usually fall (this is the 0.99 / 99th percentile). If a new reading crosses that level (i.e., it’s in the top 1% most extreme values), we mark it as 1; otherwise, it’s 0.

*This makes rare but critical spikes stand out clearly, improving the model’s ability to detect anomalies that occur infrequently.*

In [ ]:
Xn = Xn.copy()

Xn.loc[:, 'X3_extreme'] = (Xn['X3'] > Xn['X3'].quantile(0.99)).astype(int)
Xn.loc[:, 'X4_extreme'] = (Xn['X4'] > Xn['X4'].quantile(0.99)).astype(int)

# Data Transformation

Applying Log tranformation on X3 and X4 since there are exponentially exploded values. Applying log transformation will reduce skewness and stabilize variance

In [ ]:
Xn.loc[:, 'X3_log'] = np.log1p(Xn['X3'])
Xn.loc[:, 'X4_log'] = np.log1p(Xn['X4'])

# Dropping raw features
Xn = Xn.drop(columns=['X3', 'X4'])

# Scaling

In [ ]:
scale_cols = ['X1', 'X2', 'X5', 'X3_log', 'X4_log']

scaler = StandardScaler()
Xn.loc[:, scale_cols] = scaler.fit_transform(Xn[scale_cols])

# Final training feature set

In [ ]:
X_train = Xn[['X1', 'X2', 'X5', 'X3_log', 'X4_log', 'X3_extreme', 'X4_extreme']]
X_train

# Class Imbalance of Target Variable
The dataset is severely class-imbalanced, with anomalies making up <1% (~0.86%) of all observations, which is typical for real industrial systems where failures are rare. '0' refers normal sample and '1' refers to Anomaly Sample.

In [ ]:
target  = df[['target']]
target.head()

In [ ]:
df['target'].value_counts(normalize=True)

# Raw Test File

In [ ]:
test_file = r'/kaggle/input/ana-verse-2-0-h/test.parquet'
X_test_raw = pd.read_parquet(test_file)
X_test_raw

# The Final Test Dataset (X_test)

Converting the raw test file into an identical format as X_train ensures feature consistency and correct model inference, so that the model sees the same variables with the same transformations, scales, and meanings during prediction as it did during training.

In [ ]:
x3_thr = df['X3'].quantile(0.99)
x4_thr = df['X4'].quantile(0.99)

X_test = X_test_raw.copy()

X_test.loc[:, 'X3_extreme'] = (X_test['X3'] > x3_thr).astype(int)
X_test.loc[:, 'X4_extreme'] = (X_test['X4'] > x4_thr).astype(int)

X_test.loc[:, 'X3_log'] = np.log1p(X_test['X3'])
X_test.loc[:, 'X4_log'] = np.log1p(X_test['X4'])

X_test = X_test.drop(columns=['X3', 'X4'])

scale_cols = ['X1', 'X2', 'X5', 'X3_log', 'X4_log']
X_test.loc[:, scale_cols] = scaler.transform(X_test[scale_cols])

X_test = X_test[['X1', 'X2', 'X5', 'X3_log', 'X4_log', 'X3_extreme', 'X4_extreme']]

X_test

# **Model Building**

## Imports

These libraries provide tools to build the model and measure how well it detects anomalies.

In [ ]:
from catboost import CatBoostClassifier

## Prepareing Features and Target

This separates sensor readings (inputs) from the anomaly label (output) and ensures the label is in the correct numeric format.

In [ ]:
y = target.iloc[:, 0].astype(int).values
X = X_train.copy() 

## Train / Validation Split

The data is split in time order so the model learns from past data and is tested on future data, just like in real life.

In [ ]:
split = int(0.8 * len(X))
X_tr, X_val = X.iloc[:split], X.iloc[split:]
y_tr, y_val = y[:split], y[split:]

## Handling Class Imbalance

Since anomalies are rare, we calculate how much more importance the model should give to anomaly cases during training.

In [ ]:
neg, pos = np.bincount(y_tr)
scale_pos_weight = neg / pos
print(f"scale_pos_weight (train only): {scale_pos_weight:.2f}")

## Defining CatBoost Model

This sets up how the model learns patterns, including how complex it can be and how strongly it should focus on rare anomalies.

In [ ]:
cat_model = CatBoostClassifier(
    
    iterations=5000,learning_rate=0.03, depth=6,

    loss_function="Logloss", eval_metric="PRAUC",     

    class_weights=[1.0, scale_pos_weight], l2_leaf_reg=3.0, random_seed=42,

    od_type="Iter", od_wait=50,         
    
    verbose=False
)


## Training Model with Early Stopping

The model learns from training data and stops automatically when performance on new data stops improving.

In [ ]:
cat_model.fit(
    X_tr, y_tr,
    eval_set=(X_val, y_val),
    use_best_model=True
)

## Validation Probability Prediction

The model outputs how likely each validation record is to be an anomaly, instead of a simple yes/no decision.

In [ ]:
val_proba = cat_model.predict_proba(X_val)[:, 1]
val_proba

## Threshold Optimization (F1-based)

We choose the probability cut-off that gives the best balance between catching anomalies and avoiding false alarms.

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_val, val_proba)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-9)

best_idx = np.argmax(f1_scores)
best_thr = thresholds[best_idx]

print(f"Best VAL threshold: {best_thr:.4f}")
print(f"Best VAL F1: {f1_scores[best_idx]:.4f}")

## Performance Evaluation

This checks how well the model performs on unseen data using clear metrics and a confusion matrix.

In [ ]:
val_pred = (val_proba >= best_thr).astype(int)

print("\n=== CATBOOST VALIDATION PERFORMANCE ===")
print(classification_report(y_val, val_pred, digits=4))
print("Confusion Matrix (VAL):")
print(confusion_matrix(y_val, val_pred))


## Final Test Inference

The final model and chosen threshold are applied to the test data to predict anomalies in real operation.

In [ ]:
test_proba = cat_model.predict_proba(X_test)[:, 1]
test_pred  = (test_proba >= best_thr).astype(int)

print("\n=== TEST PREDICTION SUMMARY ===")
unique, counts = np.unique(test_pred, return_counts=True)
print(dict(zip(unique, counts)))
print("Predicted positive rate:", test_pred.mean())

# Similarly Building the Model using LightGBM

In [ ]:
import lightgbm as lgb

y = target.iloc[:, 0].astype(int).values
X = X_train.copy()

split = int(0.8 * len(X))
X_tr, X_val = X.iloc[:split], X.iloc[split:]
y_tr, y_val = y[:split], y[split:]

neg, pos = np.bincount(y_tr)
scale_pos_weight = neg / pos
print(f"scale_pos_weight (train only): {scale_pos_weight:.2f}")

train_set = lgb.Dataset(X_tr, label=y_tr, free_raw_data=False)
val_set   = lgb.Dataset(X_val, label=y_val, reference=train_set, free_raw_data=False)

def feval_aucpr(y_pred, dataset):
    y_true = dataset.get_label()
    return "aucpr", average_precision_score(y_true, y_pred), True

params = {
    "objective": "binary","boosting_type": "gbdt","learning_rate": 0.03,

    "scale_pos_weight": float(scale_pos_weight),

    "num_leaves": 31,"min_data_in_leaf": 200, "min_gain_to_split": 0.1,"lambda_l2": 1.0,

    "feature_fraction": 0.8,"bagging_fraction": 0.8,"bagging_freq": 1,"verbosity": -1,"seed": 42,
}

model = lgb.train(
    params=params,train_set=train_set,num_boost_round=5000,
    valid_sets=[train_set, val_set],valid_names=["train", "val"],
    feval=feval_aucpr,  #  ensures eval results exist
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, first_metric_only=True, verbose=True),
        lgb.log_evaluation(period=50)
    ]
)

print("Best iteration:", model.best_iteration)

val_proba = model.predict(X_val, num_iteration=model.best_iteration)

precision, recall, thresholds = precision_recall_curve(y_val, val_proba)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-9)

best_idx = np.argmax(f1_scores)
best_thr = thresholds[best_idx]
print(f"Best VAL threshold: {best_thr:.4f} | Best VAL F1: {f1_scores[best_idx]:.4f}")

val_pred = (val_proba >= best_thr).astype(int)

print("\n=== LIGHTGBM VALIDATION PERFORMANCE ===")
print(classification_report(y_val, val_pred, digits=4))
print("Confusion Matrix (VAL):")
cm = confusion_matrix(y_val, val_pred)
print(cm)

test_proba = model.predict(X_test, num_iteration=model.best_iteration)
test_pred  = (test_proba >= best_thr).astype(int)

print("\n=== TEST PREDICTION SUMMARY ===")
unique, counts = np.unique(test_pred, return_counts=True)
print(dict(zip(unique, counts)))
print("Predicted positive rate:", test_pred.mean())


# Similarly Building the Model using XG Boost

In [ ]:
from xgboost import XGBClassifier

y = target.iloc[:, 0].astype(int).values
X = X_train.copy()

split = int(0.8 * len(X))
X_tr, X_val = X.iloc[:split], X.iloc[split:]
y_tr, y_val = y[:split], y[split:]

neg, pos = np.bincount(y_tr)
scale_pos_weight = neg / pos
print(f"scale_pos_weight (train only): {scale_pos_weight:.2f}")

model = XGBClassifier(
    n_estimators=5000,max_depth=5,learning_rate=0.03,subsample=0.8,colsample_bytree=0.8,min_child_weight=5,gamma=0.1,reg_lambda=1.0,
    objective="binary:logistic",eval_metric="aucpr",scale_pos_weight=scale_pos_weight,random_state=42, n_jobs=-1,
    tree_method="hist",early_stopping_rounds=50,   #  here (constructor)
)

model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=False
)

best_iter = model.best_iteration + 1  
val_proba = model.predict_proba(X_val, iteration_range=(0, best_iter))[:, 1]

precision, recall, thresholds = precision_recall_curve(y_val, val_proba)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-9)
best_idx = np.argmax(f1_scores)
best_thr = thresholds[best_idx]

print(f"Best VAL threshold: {best_thr:.4f} | Best VAL F1: {f1_scores[best_idx]:.4f}")

val_pred = (val_proba >= best_thr).astype(int)
print("\n=== VALIDATION PERFORMANCE ===")
print(classification_report(y_val, val_pred, digits=4))
print("Confusion Matrix (VAL):")
print(confusion_matrix(y_val, val_pred))

test_proba = model.predict_proba(X_test, iteration_range=(0, best_iter))[:, 1]
test_pred  = (test_proba >= best_thr).astype(int)

print("\n=== TEST PREDICTION SUMMARY ===")
unique, counts = np.unique(test_pred, return_counts=True)
print(dict(zip(unique, counts)))
print("Predicted positive rate:", test_pred.mean())

# Similarly Building the Model using Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# y as int
y = target.iloc[:, 0].astype(int).values
X = X_train.copy()

# time-based split
split = int(0.8 * len(X))
X_tr, X_val = X.iloc[:split], X.iloc[split:]
y_tr, y_val = y[:split], y[split:]

rf = RandomForestClassifier(
    n_estimators=500, max_depth=12, min_samples_leaf=50, min_samples_split=100, 
    class_weight="balanced_subsample", n_jobs=-1, random_state=42
)

rf.fit(X_tr, y_tr)

val_proba = rf.predict_proba(X_val)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_val, val_proba)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-9)

best_idx = np.argmax(f1_scores)
best_thr = thresholds[best_idx]

print(f"Best RF VAL threshold: {best_thr:.4f}")
print(f"Best RF VAL F1: {f1_scores[best_idx]:.4f}")

val_pred = (val_proba >= best_thr).astype(int)

print("\n=== RF VALIDATION PERFORMANCE ===")
print(classification_report(y_val, val_pred, digits=4))
print("Confusion Matrix (VAL):")
print(confusion_matrix(y_val, val_pred))

test_proba = rf.predict_proba(X_test)[:, 1]
test_pred  = (test_proba >= best_thr).astype(int)

print("\n=== RF TEST PREDICTION SUMMARY ===")
unique, counts = np.unique(test_pred, return_counts=True)
print(dict(zip(unique, counts)))
print("Predicted positive rate:", test_pred.mean())


## **Why Catboost is the model of preference ?**

1) Among all the models, **CatBoost** has outperformed other Models, in all the metrics like Accuracy, Class 1 Precision, Class 1 Recall and Class 1 F1. That is why CatBoost was selected as model of the preference for the anamaly detection.

2) **Logistic Regression** and **SVC** were not used because the data exhibits strong non-linear relationships and complex feature interactions that linear and kernel-based models struggle to capture at this scale. **KNN** was avoided due to its high computational cost and poor performance on large, high-dimensional time-series data. A single **Decision Tree** tends to overfit heavily on rare anomalies and it is unstable 

# ***Interpretation of CatBoost Model***


#### **Precision (Class 1 – Anomaly) = 0.7204**

→ When the model raises an anomaly alert, **72% of the time it is a real anomaly**.

#### **Recall (Class 1 – Anomaly) = 0.6573**

→ The model **detects about 66% of all actual anomalies** present in the data.

#### **F1-Score (Class 1) = 0.6874**

→ This reflects a **balanced trade-off between catching anomalies and avoiding false alarms**.

#### **Precision (Class 0 – Normal) = 0.9988**

→ When the model says an operation is normal, it is **almost always correct**.

#### **Recall (Class 0 – Normal) = 0.9991**

→ The model **almost never misclassifies normal operations as anomalies**.

#### **Accuracy = 0.9979**

→ Overall correctness is very high, but **mostly driven by the abundance of normal data**.

#### **Macro Average Precision = 0.8596**

→ Average precision across both classes, showing **strong overall class balance**.

#### **Macro Average Recall = 0.8282**

→ On average, the model **recovers most cases across both normal and anomaly classes**.

#### **Macro Average F1 = 0.8432**

→ Indicates **good balanced performance** when treating both classes equally.

#### **Predicted Positive Rate (Test) = 0.62%**

→ The model flags anomalies **sparingly and conservatively**, avoiding alert fatigue.

# Submission File

In [ ]:
submission = pd.DataFrame({
    "ID": X_test_raw["ID"],     
    "target": test_pred     
})

submission

In [ ]:
submission.to_csv("submission.csv", index=False)